# Etapa 4 — Embeddings e carga no Qdrant

Transforma os chunks da etapa 3 em vetores e sobe para o Qdrant com os metadados,
pronto para a busca híbrida da etapa 5.

**Entrada:** `data/chunks/chunks_fatec_rag.jsonl` (438 chunks, 21 documentos)
**Saída:** coleção `vaar_rag` no Qdrant + `data/vocabulario_esparso.json`

## O que este notebook faz

| Etapa | O que |
|---|---|
| 1 | Carrega os chunks e audita a base |
| 2 | Descarta cabeçalho e rodapé de página (chunks de 1 a 26 tokens) |
| 3 | Gera vetores densos com **Qwen3-Embedding-0.6B** (1024 dims, normalizados) |
| 4 | Gera vetores esparsos com tokenização pt-BR (IDF calculado pelo Qdrant) |
| 5 | Cria a coleção e sobe tudo, de forma **idempotente** |
| 6 | Valida com busca densa, esparsa e **híbrida por RRF nativo** |

## Contrato com o resto do time

A função `buscar()` no final devolve a lista no formato que o
`scripts/geracao_rag_final.py` já espera:

```python
[{"titulo": ..., "ano": ..., "texto": ...}, ...]
```

Então `gerar_resposta_final(pergunta, buscar(pergunta), cliente_llm)` funciona
sem alterar o código de ninguém.

## Antes de rodar

- **GPU recomendada.** O modelo tem 0,6B de parâmetros e baixa cerca de 1,2 GB
  na primeira execução. Em CPU os 422 chunks levam vários minutos.
- No Colab: `Ambiente de execução > Alterar tipo > GPU`.

## 1. Dependências

In [ ]:
# No Colab, descomente. Em ambiente local com o requirements.txt do repo já
# instalado, pule esta célula.
# !pip install -q "sentence-transformers>=3.0" "transformers>=4.51" \
#                 "qdrant-client>=1.12" "torch>=2.1" tqdm

import sys
print("Python", sys.version.split()[0])
try:
    import torch
    print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())
except ImportError:
    print("torch ainda não instalado")

## 2. Configuração

In [ ]:
from pathlib import Path

# ── Caminhos ─────────────────────────────────────────────────────────────────
# Ajuste RAIZ se rodar fora da raiz do repositório.
RAIZ = Path.cwd()
if not (RAIZ / "scripts").exists() and (RAIZ.parent / "scripts").exists():
    RAIZ = RAIZ.parent                      # rodando de dentro de notebooks/

CAMINHO_CHUNKS = RAIZ / "data" / "chunks" / "chunks_fatec_rag.jsonl"
DIR_DADOS      = RAIZ / "data"
CAMINHO_VOCAB  = DIR_DADOS / "vocabulario_esparso.json"
DIR_DADOS.mkdir(parents=True, exist_ok=True)

# ── Modelo de embedding ──────────────────────────────────────────────────────
MODELO = "Qwen/Qwen3-Embedding-0.6B"
DIMENSAO = 1024          # nativa do modelo; truncável via MRL (32..1024)
LOTE = 8                 # suba para 32 se tiver GPU com folga

# A instrução da query PRECISA ser a mesma na carga e na consulta.
# Mudou a instrução, os vetores de query mudam e a recuperação degrada.
TAREFA = (
    "Dada uma pergunta sobre o Fundeb e a complementação VAAR, recupere os "
    "trechos de normas e notas técnicas oficiais que respondem a pergunta"
)

# ── Qdrant ───────────────────────────────────────────────────────────────────
COLECAO = "vaar_rag"

# "nuvem" -> Qdrant Cloud. A equipe inteira consulta a MESMA coleção, sem cada
#            um reprocessar 1,2 GB de modelo. É o modo do projeto.
# "local" -> arquivo em disco, sem servidor e sem rede. Útil para desenvolver
#            offline ou para o professor reproduzir sem conta nenhuma.
MODO_QDRANT = "nuvem"

CAMINHO_QDRANT = str(DIR_DADOS / "qdrant")

# ── Credenciais ──────────────────────────────────────────────────────────────
# NUNCA escreva a chave aqui. Ela fica no .env (que está no .gitignore) ou é
# digitada na hora. Chave em notebook vaza para o Git no primeiro commit.
import os

try:
    from dotenv import load_dotenv
    load_dotenv(RAIZ / ".env")
except ImportError:
    pass

QDRANT_URL = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

if MODO_QDRANT == "nuvem":
    if not QDRANT_URL:
        QDRANT_URL = input("Cluster endpoint (https://....cloud.qdrant.io): ").strip()
    if not QDRANT_API_KEY:
        from getpass import getpass
        QDRANT_API_KEY = getpass("API key do Qdrant (não aparece na tela): ").strip()
    print("endpoint:", QDRANT_URL)
    print("api key  :", f"{QDRANT_API_KEY[:12]}...{QDRANT_API_KEY[-6:]}" if QDRANT_API_KEY else "AUSENTE")

# ── Limpeza ──────────────────────────────────────────────────────────────────
MIN_TOKENS = 30          # descarta cabeçalho/rodapé de página

print("raiz do repo:", RAIZ)
print("chunks:", CAMINHO_CHUNKS, "| existe:", CAMINHO_CHUNKS.exists())

## 3. Carregar os chunks e auditar

In [ ]:
import json
from collections import Counter

chunks = []
with CAMINHO_CHUNKS.open(encoding="utf-8") as fh:
    for n, linha in enumerate(fh, 1):
        linha = linha.strip()
        if not linha:
            continue
        try:
            chunks.append(json.loads(linha))
        except json.JSONDecodeError as e:
            raise ValueError(f"{CAMINHO_CHUNKS}:{n} JSON inválido: {e}") from e

print(f"chunks carregados : {len(chunks)}")
print(f"documentos        : {len({c['document_id'] for c in chunks})}")

ids = [c["chunk_id"] for c in chunks]
duplicados = [k for k, v in Counter(ids).items() if v > 1]
print(f"chunk_id duplicado: {len(duplicados)}", "OK" if not duplicados else duplicados[:5])

tokens = [c["token_count"] for c in chunks]
tokens_ord = sorted(tokens)
mediana = tokens_ord[len(tokens_ord) // 2]
print(f"token_count       : min {min(tokens)} | mediana {mediana} | max {max(tokens)}")

print("\nmetadados disponíveis para filtro:")
for campo in ("ano", "tipo_documento", "orgao"):
    valores = Counter(c["metadata"].get(campo) for c in chunks)
    print(f"  {campo:15s} {len(valores)} valores distintos")

### Nota sobre a base

A `Lei nº 14.113/2020`, que **institui o VAAR**, aparece no
`data/fonte/fundeb_vaar_atualizado.json` mas com `texto_completo: null`, e por isso não tem
chunk nenhum. É a norma mais citada por todas as outras do corpus.

Perguntas sobre o texto da lei vão cair no "não consta no contexto". Vale
extrair essa lei e refazer o chunking dela antes da entrega.

## 4. Descartar cabeçalho e rodapé

In [ ]:
descartados = [c for c in chunks if c["token_count"] < MIN_TOKENS]
uteis       = [c for c in chunks if c["token_count"] >= MIN_TOKENS]

print(f"descartados: {len(descartados)} | mantidos: {len(uteis)}\n")
print("amostra do que saiu (cabeçalho/rodapé de página):")
for c in sorted(descartados, key=lambda x: x["token_count"])[:8]:
    amostra = c["text"].replace("\n", " ")[:58]
    print(f"  {c['token_count']:3d} tok | {amostra!r}")

# Esses trechos não respondem pergunta nenhuma e, se ficassem, competiriam por
# espaço no top-k com conteúdo real.

## 5. Vetores densos com o Qwen3-Embedding-0.6B

Três detalhes do modelo que degradam a busca **sem gerar erro**:

1. **Query e documento são assimétricos.** A query leva prefixo de instrução, o
   documento não. Por isso há dois métodos, e não um `encode()` genérico.
2. **Pooling é de último token** (EOS), não média. Exige padding à esquerda,
   senão o vetor sai do token de padding.
3. **Vetores normalizados** (norma L2 = 1), então produto interno é igual a
   cosseno. No Qdrant, use `COSINE` ou `DOT`, nunca `EUCLID`.

O repositório já tem isso encapsulado em `src/embedding/qwen.py`. O notebook usa
essa classe quando disponível, para não haver duas implementações divergindo.

In [ ]:
import numpy as np

try:
    sys.path.insert(0, str(RAIZ))
    from src.embedding import QwenEmbedder
    embedder = QwenEmbedder(model_name=MODELO, batch_size=LOTE)
    print("usando src/embedding/qwen.py do repositório")

    def vetorizar_documentos(textos):
        return embedder.encode_documents(textos)

    def vetorizar_query(texto):
        return embedder.encode_queries([texto], task=TAREFA)[0]

except Exception as e:
    print(f"src/embedding indisponível ({type(e).__name__}), usando versão local\n")
    from sentence_transformers import SentenceTransformer

    modelo = SentenceTransformer(MODELO, tokenizer_kwargs={"padding_side": "left"})

    def vetorizar_documentos(textos):
        v = modelo.encode(list(textos), batch_size=LOTE, normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=True)
        return np.asarray(v, dtype=np.float32)

    def vetorizar_query(texto):
        v = modelo.encode([texto], prompt=f"Instruct: {TAREFA}\nQuery:",
                          normalize_embeddings=True, convert_to_numpy=True)
        return np.asarray(v, dtype=np.float32)[0]

print("pronto")

In [ ]:
import time

textos = [c["text"] for c in uteis]
inicio = time.perf_counter()
vetores = vetorizar_documentos(textos)          # baixa ~1,2 GB na 1a vez
decorrido = time.perf_counter() - inicio

print(f"\nmatriz: {vetores.shape} em {decorrido:.1f}s "
      f"({decorrido / max(len(textos), 1):.2f}s por chunk)")

# Conferências que pegam erro silencioso de configuração do modelo
normas = np.linalg.norm(vetores, axis=1)
assert vetores.shape == (len(uteis), DIMENSAO), f"esperado (n, {DIMENSAO})"
assert np.allclose(normas, 1.0, atol=1e-3), "vetores não normalizados"

v_doc = vetorizar_documentos([textos[0]])[0]
v_qry = vetorizar_query(textos[0])
assert not np.allclose(v_doc, v_qry), "prefixo de instrução não foi aplicado"
print("normas OK | assimetria query/documento OK")

## 6. Vetores esparsos (lado lexical)

O denso "borra" âncoras de correspondência exata: `art. 14`, `3106200`,
`Portaria 14/2025`. O corpus é cheio delas, então o lado lexical não é opcional.

**O IDF fica no servidor.** A coleção é criada com `Modifier.IDF`, então o
cliente envia só a frequência do termo e o Qdrant calcula a raridade. Isso evita
a armadilha do BM25 Okapi, cujo IDF zera quando o termo aparece em metade dos
documentos, o que derrubaria palavras legítimas como "fundeb" e "municipio".

**O vocabulário precisa ser salvo.** O Qdrant indexa por índice inteiro, não por
palavra. Sem o mesmo mapeamento na hora da consulta, a busca esparsa não casa
nada. É o erro mais fácil de cometer aqui.

In [ ]:
import re
import unicodedata

# Mesmas regras de src/esparso/bm25.py, para os dois lados não divergirem.
STOPWORDS_PT = {
    "a", "ao", "aos", "as", "da", "das", "de", "do", "dos", "e", "em", "na",
    "nas", "no", "nos", "o", "os", "ou", "para", "pela", "pelas", "pelo",
    "pelos", "por", "que", "se", "um", "uma", "umas", "uns", "com", "como",
    "sao", "foi", "ser", "sua", "seu", "suas", "seus", "isso", "este", "esta",
    "esse", "essa", "aquele", "aquela", "qual", "quais", "quando", "onde",
}
# Aceita '.', '-' e '/' NO MEIO do token, para não picar "art.14",
# "3106200-1" nem "2025/2026", que são exatamente as âncoras que importam.
PADRAO_TOKEN = re.compile(r"[a-z0-9]+(?:[.\-/][a-z0-9]+)*")


def sem_acento(texto: str) -> str:
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def tokenizar(texto: str) -> list:
    tokens = PADRAO_TOKEN.findall(sem_acento(texto.lower()))
    return [t for t in tokens
            if (len(t) > 1 or t.isdigit()) and (t not in STOPWORDS_PT or t.isdigit())]


vocabulario: dict = {}


def vetor_esparso(texto: str, expandir: bool = True):
    '''Frequencia de termo por indice. expandir=False na consulta: termo
    novo nao existe na colecao e so geraria indice orfao.'''
    from qdrant_client import models

    contagem = {}
    for token in tokenizar(texto):
        if token in vocabulario:
            idx = vocabulario[token]
        elif expandir:
            idx = vocabulario.setdefault(token, len(vocabulario))
        else:
            continue
        contagem[idx] = contagem.get(idx, 0) + 1
    if not contagem:
        return None
    return models.SparseVector(indices=list(contagem),
                               values=[float(v) for v in contagem.values()])


esparsos = [vetor_esparso(c["text"]) for c in uteis]
vazios = sum(1 for e in esparsos if e is None)
print(f"vocabulário: {len(vocabulario):,} termos".replace(",", "."))
print(f"chunks sem token útil: {vazios}")

CAMINHO_VOCAB.write_text(json.dumps(vocabulario, ensure_ascii=False), encoding="utf-8")
print(f"vocabulário salvo em {CAMINHO_VOCAB}")
print("SEM esse arquivo a busca esparsa não funciona depois. Versione junto.")

## 7. Criar a coleção

In [ ]:
from qdrant_client import QdrantClient, models

if MODO_QDRANT == "local":
    cliente = QdrantClient(path=CAMINHO_QDRANT)
    print(f"Qdrant local em {CAMINHO_QDRANT}")
else:
    cliente = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)
    existentes = [c.name for c in cliente.get_collections().collections]
    print(f"conectado ao cluster | coleções existentes: {existentes or '(nenhuma)'}")

RECRIAR = True      # False preserva a coleção e só faz upsert por cima

if RECRIAR and cliente.collection_exists(COLECAO):
    cliente.delete_collection(COLECAO)
    print(f"coleção '{COLECAO}' anterior removida")

if not cliente.collection_exists(COLECAO):
    cliente.create_collection(
        collection_name=COLECAO,
        vectors_config={
            "denso": models.VectorParams(size=DIMENSAO, distance=models.Distance.COSINE),
        },
        sparse_vectors_config={
            # IDF calculado pelo servidor: o cliente manda só frequência de termo
            "esparso": models.SparseVectorParams(modifier=models.Modifier.IDF),
        },
    )
    print(f"coleção '{COLECAO}' criada: denso {DIMENSAO} cosine + esparso IDF")

# Índices de payload aceleram o filtro por ano, tipo e órgão.
# Na NUVEM eles valem de verdade. No modo local o cliente avisa que não têm
# efeito, e o aviso é esperado: a coleção fica pronta para quando virar servidor.
for campo, tipo in [
    ("ano", models.PayloadSchemaType.INTEGER),
    ("tipo_documento", models.PayloadSchemaType.KEYWORD),
    ("orgao", models.PayloadSchemaType.KEYWORD),
    ("document_id", models.PayloadSchemaType.KEYWORD),
]:
    try:
        cliente.create_payload_index(COLECAO, field_name=campo, field_schema=tipo)
    except Exception:
        pass
print("índices de payload configurados")

## 8. Subir os pontos

O id de cada ponto é um **UUID5 derivado do `chunk_id`**, então é estável entre
execuções: rodar o notebook de novo atualiza o mesmo ponto em vez de duplicar.
Se o id fosse posicional, qualquer reprocessamento criaria uma base inflada.

In [ ]:
import uuid

# Namespace fixo do projeto. Não mude: mudaria todos os ids.
NAMESPACE = uuid.UUID("6f1a0b3c-6d2e-4a58-9b7f-2c9d5e8a1b40")

pontos = []
for chunk, denso, esparso in zip(uteis, vetores, esparsos):
    if esparso is None:
        continue
    meta = chunk.get("metadata", {})
    pontos.append(models.PointStruct(
        id=str(uuid.uuid5(NAMESPACE, chunk["chunk_id"])),
        vector={"denso": denso.tolist(), "esparso": esparso},
        payload={
            # nomes que o scripts/geracao_rag_final.py já consome
            "titulo": chunk["title"],
            "texto": chunk["text"],
            "ano": meta.get("ano"),
            # rastreabilidade e filtro
            "chunk_id": chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "page": chunk["page"],
            "token_count": chunk["token_count"],
            "tipo_documento": meta.get("tipo_documento"),
            "orgao": meta.get("orgao"),
        },
    ))

TAMANHO_LOTE = 128
for i in range(0, len(pontos), TAMANHO_LOTE):
    cliente.upsert(collection_name=COLECAO, points=pontos[i:i + TAMANHO_LOTE])
    print(f"  {min(i + TAMANHO_LOTE, len(pontos))}/{len(pontos)}", end="\r")

total = cliente.count(COLECAO).count
print(f"\ncoleção '{COLECAO}': {total} pontos")
assert total == len(pontos), f"esperado {len(pontos)}, veio {total}"

## 9. Busca

In [ ]:
def buscar(pergunta: str, top_k: int = 5, candidatos: int = 50,
           ano=None, tipo_documento=None, com_score=False):
    '''Busca hibrida: denso + esparso fundidos por RRF nativo do Qdrant.

    Devolve [{'titulo', 'ano', 'texto', ...}], que e exatamente o formato
    esperado por scripts/geracao_rag_final.py.

    `candidatos` e quanto se pede a CADA lado antes de fundir. Precisa ser
    bem maior que top_k, senao a fusao nao tem o que reordenar.
    '''
    condicoes = []
    if ano is not None:
        condicoes.append(models.FieldCondition(key="ano", match=models.MatchValue(value=ano)))
    if tipo_documento is not None:
        condicoes.append(models.FieldCondition(
            key="tipo_documento", match=models.MatchValue(value=tipo_documento)))
    filtro = models.Filter(must=condicoes) if condicoes else None

    # ATENCAO: o filtro precisa ir DENTRO de cada prefetch. Passado so no topo
    # da consulta com fusao, ele nao propaga para as sub-consultas: cada lado
    # traz candidatos sem filtro e o RRF so reordena, entao vazam documentos de
    # outros anos. Verificado na pratica; mantenha nos dois lugares.
    prefetch = [models.Prefetch(query=vetorizar_query(pergunta).tolist(),
                                using="denso", limit=candidatos, filter=filtro)]
    esparso_q = vetor_esparso(pergunta, expandir=False)
    if esparso_q is not None:
        prefetch.append(models.Prefetch(query=esparso_q, using="esparso",
                                        limit=candidatos, filter=filtro))

    resultado = cliente.query_points(
        collection_name=COLECAO,
        prefetch=prefetch,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        query_filter=filtro,
        limit=top_k,
        with_payload=True,
    )

    saida = []
    for ponto in resultado.points:
        item = dict(ponto.payload)
        if com_score:
            item["score"] = ponto.score
        saida.append(item)
    return saida


print("buscar() pronta")

In [ ]:
PERGUNTAS = [
    "quais sao as condicionalidades para o municipio receber o VAAR",
    "como e calculado o indicador de aprendizagem com reducao das desigualdades",
    "qual o cronograma de desembolso da complementacao VAAR",
]

for pergunta in PERGUNTAS:
    print("=" * 78)
    print("?", pergunta)
    for i, doc in enumerate(buscar(pergunta, top_k=3, com_score=True), 1):
        print(f"\n  [{i}] {doc['score']:.5f}  {doc['titulo'][:62]}")
        print(f"      {doc['tipo_documento']} {doc['ano']} | pág. {doc['page']}")
        print(f"      {doc['texto'][:150].strip()}...")
    print()

### Comparando os dois lados

Vale ver onde cada retriever acerta, porque é isso que justifica a busca híbrida
no relatório do projeto.

In [ ]:
def so_denso(pergunta, k=3):
    r = cliente.query_points(COLECAO, query=vetorizar_query(pergunta).tolist(),
                             using="denso", limit=k, with_payload=True)
    return [(p.score, p.payload["titulo"][:48]) for p in r.points]


def so_esparso(pergunta, k=3):
    v = vetor_esparso(pergunta, expandir=False)
    if v is None:
        return []
    r = cliente.query_points(COLECAO, query=v, using="esparso", limit=k, with_payload=True)
    return [(p.score, p.payload["titulo"][:48]) for p in r.points]


for pergunta in ["art. 14 da Lei 14.113",
                 "o que acontece se a rede nao melhorar seus indicadores"]:
    print("=" * 78)
    print("?", pergunta)
    print("\n  DENSO (semantico)")
    for s, t in so_denso(pergunta):
        print(f"    {s:8.4f}  {t}")
    print("  ESPARSO (lexical)")
    for s, t in so_esparso(pergunta):
        print(f"    {s:8.4f}  {t}")
    print()

## 10. Filtros por metadado

In [ ]:
print("Só normas de 2026:")
for doc in buscar("condicionalidades do VAAR", top_k=3, ano=2026):
    print(f"  {doc['ano']} {doc['tipo_documento']:24s} {doc['titulo'][:46]}")

print("\nSó Resoluções:")
for doc in buscar("condicionalidades do VAAR", top_k=3, tipo_documento="Resolução"):
    print(f"  {doc['ano']} {doc['tipo_documento']:24s} {doc['titulo'][:46]}")

## 11. Integração com o pipeline do time

In [ ]:
# scripts/geracao_rag_final.py consome exatamente este formato.
#
#   from scripts.geracao_rag_final import gerar_resposta_final
#   from scripts.filtro_intencao import roteador_de_intencao
#
#   rota = roteador_de_intencao(pergunta, cliente_llm)
#   if rota["status"] == "aprovado":
#       docs = buscar(rota["pergunta"], top_k=5)
#       resposta = gerar_resposta_final(rota["pergunta"], docs, cliente_llm)
#
# Para o HyDE (scripts/geracao_hyde.py), vetorize o documento hipotético em vez
# da pergunta crua:
#
#   hipotetico = gerar_documento_hyde(pergunta, cliente_llm)
#   docs = buscar(hipotetico, top_k=5)

docs = buscar("quais condicionalidades habilitam o municipio", top_k=3)
print("chaves entregues ao gerador:", sorted(docs[0]))
print("\ncontexto formatado como o gerador faz:\n")
print("\n\n".join(f"Documento: {d['titulo']} ({d['ano']})\nTrecho: {d['texto'][:110]}..."
                  for d in docs))

## 12. Onde isso fica e o que falta

**Arquivos gerados**

| Caminho | O que é | Versionado |
|---|---|---|
| `data/vocabulario_esparso.json` | mapa termo → índice | **sim**, é obrigatório na consulta |
| `data/qdrant/` | coleção local, só no modo `"local"` | não, é reconstruível |

O vocabulário **é versionado de propósito**. A coleção na nuvem é compartilhada,
mas o mapa termo → índice não vai junto: ele mora no cliente. Se um integrante
clonar o repo sem esse arquivo e tentar consultar, a busca esparsa devolve zero
resultados sem dar erro nenhum. É a pegadinha mais fácil de cair aqui.

**Credenciais**

A chave nunca fica no notebook. Crie um `.env` na raiz do repositório:

```
QDRANT_URL=https://SEU-CLUSTER.sa-east-1-0.aws.cloud.qdrant.io
QDRANT_API_KEY=sua-chave-aqui
```

O `.env` está no `.gitignore`. Se preferir não criar arquivo, o notebook pergunta
na hora, e a chave não aparece na tela.

**Trocar entre nuvem e local**

Só mude `MODO_QDRANT` na célula de configuração. O resto do notebook é idêntico:
mesma coleção, mesmos vetores, mesma busca. Serve para desenvolver offline e
publicar na nuvem depois, sem reescrever nada.

**Pendências conhecidas**

1. **Falta a Lei 14.113/2020.** É a norma que institui o VAAR e está sem
   conteúdo no JSON de origem. Perguntas sobre o texto dela não têm resposta.
2. **16 chunks descartados** eram cabeçalho e rodapé. A origem é o chunking, que
   tem `MIN_CHUNK_TOKENS = 40` mas deixou passar chunks de 1 token.
3. **Chunks muito longos.** O maior tem 2.399 tokens, contra a meta de 600 do
   `chunking.py`. Chunk grande dilui o vetor e piora a precisão.
4. **Sem conjunto de avaliação.** Antes de ajustar `top_k` ou pesos, monte 30 a
   50 perguntas com o trecho correto anotado e meça Recall@5. Sem isso, qualquer
   ajuste é chute, e o `avaliacao_metricas.py` só mede factualidade da geração,
   não a qualidade da recuperação.